# CAL_GRN — Batch GRN generation + SERGIO simulation

This notebook:
1. Iterates over all combinations of `within_mu_level`, `inter_density`, and `max_regulator_communities`.
2. For each combo, runs **`code/1_sergio_grn_gen.py`** to generate a GRN with **1000 targets** into `./generated_data/<combo_slug>/`.
3. Then runs **`code/2_run_sergio.py`** to simulate **1000 unperturbed** and **300 perturbed** cells, saving the **`.h5ad`** next to the GRN.
4. Uses parallelism via `ThreadPoolExecutor` (safe for launching subprocesses from notebooks).

Assumes this notebook lives **alongside** the `code/` folder (not inside it).

In [7]:
from pathlib import Path
import itertools, subprocess, json, os, sys, time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Paths ---
BASE = Path('.').resolve()
CODE_DIR = BASE / 'code'
SERGIO_DIR = CODE_DIR / 'SERGIO'           # if needed by 2_run_sergio.py
GEN_SCRIPT = CODE_DIR / '1_sergio_grn_gen.py'
RUN_SCRIPT = CODE_DIR / '2_run_sergio.py'
OUT_ROOT = BASE / 'generated_data'
OUT_ROOT.mkdir(exist_ok=True)

assert GEN_SCRIPT.exists(), f"Missing: {GEN_SCRIPT}"
assert RUN_SCRIPT.exists(), f"Missing: {RUN_SCRIPT}"

# --- Grid ---
WITHIN = ['low','mid','high']
INTER = ['none','five_per_pair','half_small']
MAX_REG_COMM = ['1','3','all']
PARAM_GRID = list(itertools.product(WITHIN, INTER, MAX_REG_COMM))

# --- Fixed settings ---
N_TARGETS = 100 #XXX1000
N_UNPERTURBED = 10 #XXX1000
N_PERTURBED = 5 #XXX300
N_COMMUNITIES = 3 #XXX5  # uses default values from the generator if not passed; we pass explicitly for clarity
NOISE_PARAMS = 1
DECAYS = 0.8
SAMPLING_STATE = 15
NOISE_TYPE = 'dpd'
N_JOBS = min(4, os.cpu_count() or 1)   # tune to your machine

def slugify(within, inter, maxrc):
    return f"within-{within}__inter-{inter}__maxrc-{maxrc}"

def seed_for(within, inter, maxrc):
    s = abs(hash((within, inter, maxrc))) % (2**31 - 1)
    return int(s)

def run_one_combo(within, inter, maxrc):
    slug = slugify(within, inter, maxrc)
    out_dir = OUT_ROOT / slug
    out_dir.mkdir(parents=True, exist_ok=True)
    seed = seed_for(within, inter, maxrc)

    # 1) Generate GRN
    gen_cmd = [
        sys.executable, str(GEN_SCRIPT),
        '--out', str(out_dir),
        '--seed', str(seed),
        '--communities', str(N_COMMUNITIES),
        '--within', within,
        '--inter', inter,
        '--n_targets', str(N_TARGETS),
        '--max_regulator_communities', maxrc
    ]
    t0 = time.time()
    gen = subprocess.run(gen_cmd, capture_output=True, text=True)
    t1 = time.time()
    gen_ok = (gen.returncode == 0)

    # 2) Run SERGIO driver only if generation succeeded
    h5ad_path = out_dir / 'adata.h5ad'
    if gen_ok:
        run_cmd = [
            sys.executable, str(RUN_SCRIPT),
            '--grn_dir', str(out_dir),
            '--out', str(h5ad_path),
            '--n_unperturbed', str(N_UNPERTURBED),
            '--n_perturbed', str(N_PERTURBED),
            '--seed', str(seed + 1337),
            '--sergio_dir', str(SERGIO_DIR),
            '--noise_params', str(NOISE_PARAMS),
            '--decays', str(DECAYS),
            '--sampling_state', str(SAMPLING_STATE),
            '--noise_type', str(NOISE_TYPE)
        ]
        sim = subprocess.run(run_cmd, capture_output=True, text=True)
        sim_ok = (sim.returncode == 0)
        return {
            'slug': slug, 'out_dir': str(out_dir),
            'gen_ok': gen_ok, 'gen_secs': t1 - t0, 'gen_stdout': gen.stdout, 'gen_stderr': gen.stderr,
            'sim_ok': sim_ok, 'sim_secs': (time.time() - t1), 'sim_stdout': sim.stdout, 'sim_stderr': sim.stderr,
            'h5ad': str(h5ad_path) if sim_ok else None,
            'within': within, 'inter': inter, 'max_reg_comm': maxrc
        }
    else:
        return {
            'slug': slug, 'out_dir': str(out_dir),
            'gen_ok': gen_ok, 'gen_secs': t1 - t0, 'gen_stdout': gen.stdout, 'gen_stderr': gen.stderr,
            'sim_ok': False, 'sim_secs': 0.0, 'sim_stdout': '', 'sim_stderr': 'generator_failed',
            'h5ad': None,
            'within': within, 'inter': inter, 'max_reg_comm': maxrc
        }

print(f"Total combos: {len(PARAM_GRID)} — running with N_JOBS={N_JOBS}")


Total combos: 27 — running with N_JOBS=4


In [10]:
results = []
with ThreadPoolExecutor(max_workers=N_JOBS) as ex:
    futs = {ex.submit(run_one_combo, w, i, m): (w, i, m) for (w,i,m) in PARAM_GRID}
    for fut in as_completed(futs):
        w,i,m = futs[fut]
        try:
            res = fut.result()
        except Exception as e:
            res = {'slug': f'{w}-{i}-{m}', 'gen_ok': False, 'sim_ok': False, 'error': str(e)}
        results.append(res)
len(results)


27

In [9]:
df = pd.DataFrame(results)
display(df[['slug','gen_ok','sim_ok','out_dir','h5ad','within','inter','max_reg_comm','gen_secs','sim_secs']])
df.to_csv(OUT_ROOT / 'batch_summary.csv', index=False)
print('Saved summary to', OUT_ROOT / 'batch_summary.csv')


,slug,gen_ok,sim_ok,out_dir,h5ad,within,inter,max_reg_comm,gen_secs,sim_secs
0,within-low__inter-five_per_pair__maxrc-1,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,five_per_pair,1,0.427932,5.095806
1,within-low__inter-none__maxrc-all,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,none,all,0.464710,5.370037
2,within-low__inter-none__maxrc-3,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,none,3,0.467698,5.460267
3,within-low__inter-none__maxrc-1,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,none,1,0.485638,5.606896
4,within-low__inter-five_per_pair__maxrc-3,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,five_per_pair,3,0.395637,5.445707
5,within-low__inter-half_small__maxrc-1,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,half_small,1,0.504585,5.063068
6,within-low__inter-half_small__maxrc-3,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,half_small,3,0.451360,5.143379
7,within-low__inter-five_per_pair__maxrc-all,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,five_per_pair,all,0.467250,5.808437
8,within-low__inter-half_small__maxrc-all,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,low,half_small,all,0.458052,4.979382
9,within-mid__inter-none__maxrc-1,True,True,/home/laganiv/Desktop/projects/CausalEmbed/grn...,/home/laganiv/Desktop/projects/CausalEmbed/grn...,mid,none,1,0.444155,5.230017


Saved summary to /home/laganiv/Desktop/projects/CausalEmbed/grn_crl/refactoring/simulation/generated_data/batch_summary.csv


### Tips
- Increase `N_JOBS` to use more CPU if your machine can handle it.
- If you need to **resume**, you can add a check in `run_one_combo` to skip when `sim_all.h5ad` already exists.
- To change the number of TF communities or other generator args, expose them near the top and pass via `gen_cmd`.